# Storage Menage Repository

Esta é a documentação didática para a classe **StorageRepository**, um componente essencial para o gerenciamento híbrido de arquivos em sua aplicação.

---

## 1. Visão Geral

A classe `StorageRepository` atua como uma **camada de abstração** (ou Repositório) para o armazenamento de imagens. O seu principal objetivo é unificar dois mundos: o **armazenamento local** (disco rígido do servidor) e o **armazenamento em nuvem** (Supabase Storage).

Ao utilizar esta classe, o restante do seu código não precisa se preocupar com detalhes técnicos como manipulação de caminhos de arquivos ou protocolos de rede; ele apenas solicita que uma imagem seja salva, e o repositório decide como e onde isso deve acontecer.

---

## 2. Fluxo de Execução

O ciclo de vida de uma operação comum nesta classe segue estes passos:

1. **Configuração**: O objeto é criado recebendo um `base_path` (onde salvar localmente) e um `bucket_name` (onde salvar na nuvem).
2. **Preparação de Dados**: Ao receber uma lista de objetos `GeneratedImage`, a classe extrai os bytes e identifica o formato (MIME type).
3. **Persistência Local**: Se o método `local_repository` for chamado, a classe gera nomes únicos baseados na data e hora atual para evitar conflitos de arquivos e grava os bytes no disco.
4. **Sincronização Remota**: Se o método `upload_to_supabase` for chamado, a classe utiliza internamente o `StorageManager` para enviar os dados para a nuvem e recupera instantaneamente o link público de acesso.

---

## 3. Tabela de Métodos

| **Método** | **Tipo** | **Descrição Breve** |
| --- | --- | --- |
| `__init__` | Construtor | Define as rotas base e o bucket de destino. |
| `local_repository` | Público | Salva múltiplas imagens no sistema de arquivos local. |
| `_mime_to_extension` | Privado (Auxiliar) | Converte tipos como `image/png` para a extensão `.png`. |
| `upload_to_supabase` | Público | Envia um arquivo para o Supabase e retorna sua URL. |

---

## 4. Arquitetura e Insights

- **Padrão Repository**: Esta classe isola a lógica de infraestrutura. Se amanhã você decidir trocar o Supabase pelo Amazon S3, você só precisará alterar este arquivo, sem quebrar o resto do sistema.
- **Geração de Nomes Únicos**: O uso de `datetime.utcnow()` no salvamento local garante que, mesmo que duas imagens sejam geradas em sequência, elas tenham nomes distintos, prevenindo a sobreposição acidental de dados.
- **Tratamento de Extensões**: O método `_mime_to_extension` atua como um guardião de integridade, garantindo que apenas tipos de imagem suportados (JPEG e PNG) sejam processados, evitando arquivos corrompidos ou desconhecidos.

---

## 5. Detalhamento da Classe

### Classe StorageRepository

**Descrição**

Gerencia a persistência de imagens de forma híbrida. Permite que imagens geradas pela aplicação sejam armazenadas tanto em diretórios locais quanto no serviço de Storage do Supabase, padronizando o acesso aos arquivos.

**Argumentos**

- `base_path` (str): Diretório base local ou prefixo de pasta no storage remoto.
- `bucket_name` (str): Nome do bucket configurado no console do Supabase.

---

### Métodos

### 1. local_repository

**Descrição**

Itera sobre uma lista de imagens, gera nomes de arquivo baseados em carimbo de data/hora (timestamp) e as salva no disco local.

**Argumentos**

- `images` (List[GeneratedImage]): Lista de objetos contendo os bytes e o MIME type.
- `prefix` (str): Nome inicial do arquivo (ex: "avatar", "post"). Default é "image".

**Retornos**

- `List[str]`: Uma lista contendo o caminho completo (path) de cada arquivo salvo com sucesso.

**Raises**

- `ValueError`: Se uma imagem possuir um MIME type não mapeado.
- `IOError`: Caso ocorra erro de permissão ou espaço em disco ao gravar.

**Exemplos**

Python

# 

`repo = StorageRepository(base_path="./data/exports")
paths = repo.local_repository(lista_de_imagens, prefix="geracao_ia")
# Retorno: ["./data/exports/geracao_ia_20240101_120000_1.jpg", ...]`

---

### 2. upload_to_supabase

**Descrição**

Faz a ponte entre a aplicação e o Supabase. Envia os bytes brutos para o bucket e retorna o link direto para visualização.

**Argumentos**

- `file_name` (str): Nome que o arquivo terá dentro do bucket.
- `byte_data` (bytes): O conteúdo binário da imagem.

**Retornos**

- `str`: A URL pública (HTTP) para acessar o arquivo na nuvem.

**Raises**

- `ConnectionError`: Se houver falha na rede ou nas credenciais do Supabase.
- `ValueError`: Se os parâmetros de nome de arquivo forem inválidos.

**Exemplos**

Python

# 

`repo = StorageRepository(base_path="galeria", bucket_name="bucket_oficial")
url_publica = repo.upload_to_supabase("foto_perfil.png", dados_em_bytes)
print(url_publica) 
Saída: https://projeto.supabase.co/storage/v1/object/public/bucket_oficial/galeria/foto_perfil.png`

---

### 3. _mime_to_extension

**Descrição**

Método interno de suporte que traduz o tipo MIME da imagem para uma extensão de arquivo amigável ao sistema operacional.

**Argumentos**

- `mime_type` (str): O tipo MIME (ex: "image/jpeg").

**Retornos**

- `str`: A extensão curta (ex: "jpg" ou "png").

**Raises**

- `ValueError`: Se o MIME type não for "image/jpeg" ou "image/png".